In [ ]:
import glob 
import os 
latent_path = "/data1/syliu/ixi_mcx_2025/ixi_mcx_2025_lowres"
latents = glob.glob(os.path.join(latent_path,"*full.pt"))
paired_latents = [{"image":x,"image_source":x.replace("full.pt","simple.pt")} for x in latents]
len(paired_latents)

In [ ]:
from rectified_flow_pytorch.rectified_flow import *
import os

# 验证CUDA_HOME设置
print(f"CUDA_HOME环境变量: {os.environ.get('CUDA_HOME', '未设置')}")
if os.path.exists(os.environ.get('CUDA_HOME', '')):
    print("CUDA_HOME路径存在")
    os.system('ls $CUDA_HOME')
else:
    print("警告: CUDA_HOME路径不存在!")

# 验证CUDA是否可用
import torch
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA设备数量: {torch.cuda.device_count()}")
    print(f"当前CUDA设备: {torch.cuda.current_device()}")
    print(f"设备名称: {torch.cuda.get_device_name(torch.cuda.current_device())}")

    # 验证CUDA_VISIBLE_DEVICES设置
    print(f"CUDA_VISIBLE_DEVICES环境变量: {os.environ.get('CUDA_VISIBLE_DEVICES', '未设置')}")


In [ ]:


import monai.data as md 
from tqdm.notebook import tqdm 

import torch 
import monai

from torch.utils.data import Dataset
import glob 
import os 
import torch 
scale = 1 
def load_func(x):
    x = torch.load(x,weights_only=False)[0]
    x = x.float()/scale
    x = x *0.25
    return x
import monai.data as md  

class CT_VESSEL_AVN(Dataset):
    def __init__(self,latent_dir):
        self.latent_dir = latent_dir
        self.files = sorted(glob.glob(os.path.join(latent_dir,"*cropped_A.pt")))
        self.image_dicts=[{
            "image":file,
            "source_image":file.replace("cropped_A","cropped_N")
        } for file in self.files]
    def __len__(self):
        return len(self.image_dicts)
    def __getitem__(self,idx):
        image_dict = self.image_dicts[idx]
        return {
            "image":load_func(image_dict["image"]),
            "source_image":load_func(image_dict["source_image"]),
        }

class IXI_MCX_paired(Dataset):
    def __init__(self,latent_dir):
        self.latent_dir = latent_dir
        self.files = sorted(glob.glob(os.path.join(latent_dir,"*full.pt")))
        self.image_dicts=[{
            "image":file,
            "source_image":file.replace("full.pt","simple.pt")
        } for file in self.files]
    def __len__(self):
        return len(self.image_dicts)
    def __getitem__(self,idx):
        image_dict = self.image_dicts[idx]
        return {
            "image":load_func(image_dict["image"]),
            "source_image":load_func(image_dict["source_image"]),
        }

# latent_dir="/mnt/users/notebooks/1/2/maisi_pretrain_finetune/int8latent"
# dataset =CT_VESSEL_AVN(latent_dir)
latent_dir = 'ixi_mcx_2025_latent'
dataset = IXI_MCX_paired(latent_dir)
data_one = dataset[0]
for k,v in data_one.items(): print(k,v.shape)

In [ ]:



from monai.bundle import ConfigParser
config = ConfigParser()
config.read_config('shiyu_utils/config_maisi3d-rflow.json')
config["autoencoder_def"]["num_splits"]=1
autoencoder =config.get_parsed_content('autoencoder_def',instanitiate=True)
autoencoder.load_state_dict(torch.load("models/autoencoder_epoch273.pt"))
autoencoder.eval()


def test_latent_and_ae():
    latents = sorted(glob.glob(os.path.join(latent_dir,"*.pt")))
    latent = latents[0]
    latent = torch.load(latent,weights_only=False)
    latent = latent.float()
    latent = latent
    latent  = latent / scale 
    latent = latent.float()
    with torch.no_grad(),torch.cuda.amp.autocast(True):
        autoencoder.cuda()
        recon = autoencoder.decode_stage_2_outputs(latent.cuda())
    from shiyu_utils.plot_3d_data import plot_3d_data
    plot_3d_data(recon)
    print(latent.shape,latent.dtype)    
    # =================== 
    latents = sorted(glob.glob(os.path.join(latent_dir,"*.pt")))
    latent = latents[0]
    latent = torch.load(latent,weights_only=False)
    latent = (latent/32).to(torch.float)
    with torch.no_grad(),torch.cuda.amp.autocast(True):
        autoencoder.cuda()
        recon = autoencoder.decode_stage_2_outputs(latent.cuda())
    from shiyu_utils.plot_3d_data import plot_3d_data
    plot_3d_data(recon)
    print(latent.shape,latent.dtype)
    # ==================== 

#test_latent_and_ae()


# In[5]:



from errno import ESTALE
from rectified_flow_pytorch import RectifiedFlow, Unet, Trainer
from monai.networks.nets import DiffusionModelUNet
from torch import nn, pi, cat, stack, from_numpy
from einops import rearrange
class monai_wrapper(torch.nn.Module):
    def __init__(self,model,mean_variance_net=False):
        super().__init__()
        self.model = model
        self.mean_variance_net = mean_variance_net

    def forward(self,x,times,cond=None):
        context = cond 
        timesteps = ((1.- times) * 1000 ).long()
        #for v in[x,context,timesteps]:print(v.shape) if hasattr(v,"shape") else print("v has no shape")
        out= self.model(x,context=context,timesteps=timesteps)
        if self.mean_variance_net:
            mean, log_var = rearrange(out, 'b (c mean_log_var) h w d -> mean_log_var b c h w d', mean_log_var = 2)
            variance = log_var.exp() # variance needs to be positive
            return stack((mean,variance))
        else: 
            return out 


# In[6]:


from rectified_flow_pytorch import RectifiedFlow, Unet, Trainer

monai_model = DiffusionModelUNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    num_res_blocks=(2,2,2),
    channels=(64,64,128),
    attention_levels=(False,False,True),
    norm_num_groups=32,
    num_head_channels=64,
    cross_attention_dim=None,
    with_conditioning=False,
    use_flash_attention=True,
    )

model = monai_wrapper(monai_model,mean_variance_net=False)
rectified_flow = RectifiedFlow(model,mean_variance_net=False,data_shape=(4,64,64,64),immiscible=False)

In [6]:

from torch.optim import Adam
from accelerate import Accelerator
from torch.utils.data import DataLoader
from ema_pytorch import EMA

def cycle(dl):
    while True:
        for batch in dl:
            yield batch

from tqdm import tqdm 
from rectified_flow_pytorch import Trainer
class MyTrainer(Trainer):
    def _save_2d_in_png(self,sampled,fname):
        sampled = rearrange(sampled, '(row col) c h w -> c (row h) (col w)', row = self.num_sample_rows)
        sampled.clamp_(0., 1.)

        save_image(sampled, fname)
        return sampled

    def _save_3d_in_png(self,data,fname):
        _,_,h,w,d = data.shape
        hh,ww,dd = h//2,w//2,d//2
        data_xy=data[:,:,hh,:,:]
        data_yz=data[:,:,:,ww,:]
        data_xz=data[:,:,:,:,dd]
        fname_xy=fname.replace(".png","_xy.png")
        fname_yz=fname.replace(".png","_yz.png")
        fname_xz=fname.replace(".png","_xz.png")
        sampled_xy= self._save_2d_in_png(data_xy,fname_xy)
        sampled_yz= self._save_2d_in_png(data_yz,fname_yz)
        sampled_xz= self._save_2d_in_png(data_xz,fname_xz)
        return sampled_xy,sampled_yz,sampled_xz

    def _process_input(self,data):
        noise,cond=None,None
        if isinstance(data,tuple) or isinstance(data,list):
            data,cond=data[0],data[1]
        elif isinstance(data,dict):
            data,noise=data["image"],data["source_image"]
        return data,noise,cond
    
    def sample(self, fname):
        eval_model = default(self.ema_model, self.model)
        dl = cycle(self.dl)
        mock_data = next(dl)
        mock_data,mock_noise,mock_cond =self._process_input(mock_data)
        data_shape = mock_data.shape[1:]
        mock_noise = mock_noise.repeat(self.num_samples//mock_noise.shape[0],1,1,1,1) if hasattr(mock_noise,"shape") else None
        mock_cond = mock_cond.repeat(self.num_samples//mock_cond.shape[0],1,1) if hasattr(mock_cond,"shape") else None
        additional_sample_kwargs = dict()
        if isinstance(eval_model.model, RectifiedFlow):
            additional_sample_kwargs.update(temperature = self.sample_temperature)
            # additional_sample_kwargs.update(noise = mock_noise)
            # additional_sample_kwargs.update(cond = mock_cond)

        with torch.no_grad():
            sampled = eval_model.sample(
                batch_size = self.num_samples,
                data_shape = data_shape,
                noise = mock_noise,
                cond = mock_cond, 
                **additional_sample_kwargs
            )
            global autoencoder
            autoencoder = autoencoder.to(trainer.accelerator.device)
            sampled_collect=[]
            with torch.cuda.amp.autocast(True):
                for sample_per_batch in tqdm(sampled):
                    sample_per_batch = autoencoder.decode_stage_2_outputs(sample_per_batch[None,...]/0.25)
                    sampled_collect.append(sample_per_batch)
                sampled = torch.cat(sampled_collect,dim=0)
            sample_output = self._save_3d_in_png(sampled,fname)

            mock_data_collect=[]
            with torch.cuda.amp.autocast(True):
                for sample_per_batch in tqdm(mock_data):
                    sample_per_batch = autoencoder.decode_stage_2_outputs(sample_per_batch[None,...]/0.25)
                    mock_data_collect.append(sample_per_batch)
                mock_data = torch.cat(mock_data_collect,dim=0)
            sample_output = self._save_3d_in_png(mock_data,fname.replace(".png","_tar.png"))
            mock_noise_collect=[]
            if mock_noise is not None:
                with torch.cuda.amp.autocast(True):
                    for sample_per_batch in tqdm(mock_noise):
                        sample_per_batch = autoencoder.decode_stage_2_outputs(sample_per_batch[None,...]/0.25)
                        mock_noise_collect.append(sample_per_batch)
                    mock_noise = torch.cat(mock_noise_collect,dim=0)
                mock_noise = self._save_3d_in_png(mock_noise,fname.replace(".png","_src.png"))
                return sample_output,mock_noise
            else:
                return sample_output
    def forward(self):

        dl = cycle(self.dl)

        for ind in range(self.num_train_steps):
            step = ind + 1

            self.model.train()

            data = next(dl)

            data,noise,cond=self._process_input(data)
            if self.return_loss_breakdown:
                loss, loss_breakdown = self.model(data,noise=noise,cond=cond, return_loss_breakdown = True)
                self.log(loss_breakdown._asdict(), step = step)
            else:
                loss = self.model(data)

            self.accelerator.print(f'[{step}] loss: {loss.item():.3f}')
            self.accelerator.backward(loss)

            self.accelerator.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)

            self.optimizer.step()
            self.optimizer.zero_grad()

            if getattr(self.model, 'use_consistency', False):
                self.model.ema_model.update()

            if self.is_main and self.use_ema:
                self.ema_model.ema_model.data_shape = self.model.data_shape
                self.ema_model.update()

            self.accelerator.wait_for_everyone()
            if self.is_main:

                if divisible_by(step, self.save_results_every) or step==1: # we want to sample for the first step to debug sample 

                    sampled = self.sample(fname = str(self.results_folder / f'results.{step}.png'))

                    self.log_images(sampled, step = step)

                if divisible_by(step, self.checkpoint_every) or step==1:
                    self.save(f'checkpoint.{step}.pt')

            self.accelerator.wait_for_everyone()

        print('training complete')
trainer = MyTrainer(
    rectified_flow,
    dataset = dataset,
    batch_size=4,
    num_samples=4,
    num_train_steps = 70_000,
    sample_temperature = 1.5,
    checkpoint_every=10000,
    checkpoints_folder="./checkpoints_latent_simple2full",
    results_folder = './results_latent_simple2full',  # samples will be saved periodically to this folder
)

trainer()

[63484] loss: 0.130
[63485] loss: 0.234
[63486] loss: 0.157
[63487] loss: 0.139
[63488] loss: 0.184
[63489] loss: 0.170
[63490] loss: 0.120
[63491] loss: 0.253
[63492] loss: 0.182
[63493] loss: 0.222
[63494] loss: 0.188
[63495] loss: 0.105
[63496] loss: 0.133
[63497] loss: 0.142
[63498] loss: 0.130
[63499] loss: 0.157
[63500] loss: 0.214


100%|██████████| 4/4 [00:07<00:00,  1.97s/it]


[63501] loss: 0.149
[63502] loss: 0.093
[63503] loss: 0.144
[63504] loss: 0.203
[63505] loss: 0.126
[63506] loss: 0.176
[63507] loss: 0.159
[63508] loss: 0.205
[63509] loss: 0.241
[63510] loss: 0.202
[63511] loss: 0.176
[63512] loss: 0.088
[63513] loss: 0.192
[63514] loss: 0.150
[63515] loss: 0.180
[63516] loss: 0.195
[63517] loss: 0.168
[63518] loss: 0.155
[63519] loss: 0.132
[63520] loss: 0.170
[63521] loss: 0.155
[63522] loss: 0.097
[63523] loss: 0.144
[63524] loss: 0.202
[63525] loss: 0.189
[63526] loss: 0.203
[63527] loss: 0.246
[63528] loss: 0.193
[63529] loss: 0.194
[63530] loss: 0.169
[63531] loss: 0.183
[63532] loss: 0.220
[63533] loss: 0.156
[63534] loss: 0.134
[63535] loss: 0.116
[63536] loss: 0.181
[63537] loss: 0.140
[63538] loss: 0.159
[63539] loss: 0.219
[63540] loss: 0.208
[63541] loss: 0.172
[63542] loss: 0.191
[63543] loss: 0.187
[63544] loss: 0.231
[63545] loss: 0.116
[63546] loss: 0.234
[63547] loss: 0.137
[63548] loss: 0.186
[63549] loss: 0.164
[63550] loss: 0.177


100%|██████████| 4/4 [00:08<00:00,  2.03s/it]


[63601] loss: 0.172
[63602] loss: 0.190
[63603] loss: 0.180
[63604] loss: 0.166
[63605] loss: 0.141
[63606] loss: 0.154
[63607] loss: 0.192
[63608] loss: 0.161
[63609] loss: 0.208
[63610] loss: 0.195
[63611] loss: 0.161
[63612] loss: 0.168
[63613] loss: 0.149
[63614] loss: 0.174
[63615] loss: 0.153
[63616] loss: 0.164
[63617] loss: 0.175
[63618] loss: 0.187
[63619] loss: 0.160
[63620] loss: 0.160
[63621] loss: 0.102
[63622] loss: 0.170
[63623] loss: 0.121
[63624] loss: 0.140
[63625] loss: 0.157
[63626] loss: 0.181
[63627] loss: 0.206
[63628] loss: 0.161
[63629] loss: 0.164
[63630] loss: 0.229
[63631] loss: 0.182
[63632] loss: 0.115
[63633] loss: 0.140
[63634] loss: 0.164
[63635] loss: 0.154
[63636] loss: 0.176
[63637] loss: 0.164
[63638] loss: 0.093
[63639] loss: 0.122
[63640] loss: 0.214
[63641] loss: 0.183
[63642] loss: 0.191
[63643] loss: 0.196
[63644] loss: 0.112
[63645] loss: 0.132
[63646] loss: 0.197
[63647] loss: 0.138
[63648] loss: 0.119
[63649] loss: 0.154
[63650] loss: 0.167


...

[69801] loss: 0.207
[69802] loss: 0.126
[69803] loss: 0.192
[69804] loss: 0.191
[69805] loss: 0.127
[69806] loss: 0.166
[69807] loss: 0.186
[69808] loss: 0.136
[69809] loss: 0.128
[69810] loss: 0.182
[69811] loss: 0.176
[69812] loss: 0.167
[69813] loss: 0.217
[69814] loss: 0.158
[69815] loss: 0.121
[69816] loss: 0.186
[69817] loss: 0.103
[69818] loss: 0.177
[69819] loss: 0.219
[69820] loss: 0.158
[69821] loss: 0.125
[69822] loss: 0.174
[69823] loss: 0.188
[69824] loss: 0.178
[69825] loss: 0.209
[69826] loss: 0.208
[69827] loss: 0.116
[69828] loss: 0.215
[69829] loss: 0.138
[69830] loss: 0.150
[69831] loss: 0.141
[69832] loss: 0.159
[69833] loss: 0.141
[69834] loss: 0.164
[69835] loss: 0.098
[69836] loss: 0.244
[69837] loss: 0.106
[69838] loss: 0.221
[69839] loss: 0.186
[69840] loss: 0.195
[69841] loss: 0.164
[69842] loss: 0.212
[69843] loss: 0.135
[69844] loss: 0.177
[69845] loss: 0.178
[69846] loss: 0.159
[69847] loss: 0.207
[69848] loss: 0.236
[69849] loss: 0.202
[69850] loss: 0.113


100%|██████████| 4/4 [00:08<00:00,  2.02s/it]


[69901] loss: 0.107
[69902] loss: 0.261
[69903] loss: 0.124
[69904] loss: 0.171
[69905] loss: 0.186
[69906] loss: 0.191
[69907] loss: 0.169
[69908] loss: 0.184
[69909] loss: 0.119
[69910] loss: 0.225
[69911] loss: 0.199
[69912] loss: 0.222
[69913] loss: 0.177
[69914] loss: 0.170
[69915] loss: 0.117
[69916] loss: 0.244
[69917] loss: 0.138
[69918] loss: 0.128
[69919] loss: 0.181
[69920] loss: 0.222
[69921] loss: 0.207
[69922] loss: 0.232
[69923] loss: 0.089
[69924] loss: 0.186
[69925] loss: 0.180
[69926] loss: 0.245
[69927] loss: 0.087
[69928] loss: 0.204
[69929] loss: 0.156
[69930] loss: 0.226
[69931] loss: 0.175
[69932] loss: 0.165
[69933] loss: 0.184
[69934] loss: 0.162
[69935] loss: 0.163
[69936] loss: 0.192
[69937] loss: 0.153
[69938] loss: 0.129
[69939] loss: 0.198
[69940] loss: 0.192
[69941] loss: 0.136
[69942] loss: 0.158
[69943] loss: 0.149
[69944] loss: 0.191
[69945] loss: 0.183
[69946] loss: 0.129
[69947] loss: 0.113
[69948] loss: 0.197
[69949] loss: 0.175
[69950] loss: 0.159


100%|██████████| 4/4 [00:07<00:00,  2.00s/it]


training complete


In [7]:
1

1